# Profiling Struphy with scope-profiler

This tutorial shows two ways to use `scope-profiler`:

1. Profile a small standalone Python workload with `ProfileManager` region timing.
2. Configure a Struphy `Simulation` with `ProfilingOptions`, including line profiling, and activate profiling at `sim.run(...)`.

The same HDF5 output file can be inspected from Python, from the command-line interface, or in the terminal TUI.

## Imports and output directory

Use a small temporary directory so repeated notebook runs do not overwrite production simulation data.

In [ ]:
from pathlib import Path
import shutil
import tempfile

import numpy as np
from scope_profiler import ProfileManager

workdir = Path(tempfile.mkdtemp(prefix="prof_", dir="."))


In [ ]:
from contextlib import redirect_stderr, redirect_stdout
from html import escape
from io import StringIO

from IPython.display import HTML, display


def show_full_text(text: str):
    """Render long text as HTML so notebook frontends do not truncate stream output."""
    display(
        HTML(
            '<pre style="white-space: pre-wrap; overflow: visible; max-height: none; '
            'font-family: var(--jp-code-font-family, monospace);">'
            f"{escape(text)}"
            "</pre>"
        )
    )


def capture_full_output(func, *args, **kwargs):
    stdout = StringIO()
    stderr = StringIO()
    with redirect_stdout(stdout), redirect_stderr(stderr):
        result = func(*args, **kwargs)
    text = stderr.getvalue() + stdout.getvalue()
    if text:
        show_full_text(text)
    return result


## 1. Profile standalone Python code

`ProfileManager.profile_region(...)` profiles a `with` block. `@ProfileManager.profile(...)` profiles a decorated function. This first example only records region timings, which is enough for coarse timing and produces a small HDF5 profile file.

In [ ]:
@ProfileManager.profile("demo: matrix multiply")
def matrix_work(n: int) -> float:
    a = np.arange(n * n, dtype=float).reshape(n, n)
    b = a.T.copy()
    c = a @ b
    return float(c.sum())


profile_file = workdir / "demo_scope_profile.h5"

with ProfileManager.session(
    file_path=str(profile_file),
    return_results=True,
    verbose=False,
) as prof:
    with ProfileManager.profile_region("demo: total"):
        total = 0.0
        for n in (16, 24, 32):
            total += matrix_work(n)
        print(f"Total sum: {total:.2f}")

results = prof.results

The context manager finalizes the run and writes one HDF5 file. With `return_results=True`, the finalized result object is also available in memory.

In [ ]:
results.print_summary(title="Standalone demo", include=[r"^demo:"])

The same file can be inspected from Python. The CLI and TUI commands are shown later for terminal workflows.


In [ ]:
from scope_profiler import read_h5

standalone_results = read_h5(profile_file)
standalone_results.print_summary(title="Standalone HDF5 summary", include=[r"^demo:"])


## 2. Profile a Struphy simulation

Struphy simulations use the same `ProfileManager` infrastructure internally. Configure profiler details with `ProfilingOptions`, pass them to `Simulation`, and activate profiling when calling `sim.run(...)`.

This example enables `use_line_profiler=True` for the Struphy run. Line profiling has more overhead than region timing, so it is best used on small reproducer cases or short runs.

In [ ]:
from struphy import EnvironmentOptions, ProfilingOptions, Simulation, Time, grids
from struphy.models import Poisson

env = EnvironmentOptions(
    out_folders=str(workdir),
    sim_folder="poisson_profile_demo",
    save_step=1,
)

profiling_opts = ProfilingOptions(
    file_path=str(workdir / "poisson_profile.h5"),
    use_line_profiler=True,
)

sim = Simulation(
    model=Poisson(),
    env=env,
    time_opts=Time(dt=0.01, Tend=0.01),
    grid=grids.TensorProductGrid(num_elements=(4, 4, 1)),
    profiling_opts=profiling_opts,
)

In [ ]:
# Run this cell when you want to create the Struphy profiling file.
# capture_full_output(sim.run, profiling_activated=True)
sim.run(profiling_activated=True)
struphy_profile_file = Path(profiling_opts.file_path)


In [ ]:
from scope_profiler import read_h5

profile_results = read_h5(struphy_profile_file)
region_names = sorted(region.name for region in profile_results.get_regions())
print(", ".join(region_names))

### Line-profile output for selected Struphy regions

Because the Struphy run used `use_line_profiler=True`, the HDF5 file contains line-by-line timing records for decorated functions and profiled regions. The filter below prints two representative regions:

- `setup: total`, the top-level setup block in `Simulation.run`.
- `solve: PoissonSolve`, the Poisson propagator solve call.

The source column is shown because these regions come from real Struphy source files.

In [ ]:
from scope_profiler.line_profile_cli import print_line_profile

print_line_profile(
    struphy_profile_file,
    region=r"^prop: PoissonSolve",
    display_html=True,
)

For a parameter file generated by `struphy params <ModelName>`, the same pattern is:

```python
profiling_opts = ProfilingOptions(
    file_path="my_profile.h5",
    use_line_profiler=True,
)

sim = Simulation(..., profiling_opts=profiling_opts)
sim.run(profiling_activated=True)
```

Omit the `profiling_activated` keyword for normal production runs without profiling overhead.

## 3. Plot and inspect the profile

`scope-profiler` also provides a CLI and a terminal TUI for interactive work outside the notebook:

```bash
scope-profiler inspect profiling_data.h5
scope-profiler line-profile profiling_data.h5
scope-profiler plot quick profiling_data.h5 -o figures
scope-profiler tui profiling_data.h5
```

We recommend checking out the [scope-profiler documentation](https://max-models.github.io/scope-profiler/) for details on postprocessing using the tool.

Inside a notebook, it is often more convenient to use the Python API directly. The cells below load the HDF5 profile with `read_h5` and build figures with `plot_durations`, `plot_gantt`, and `plot_flame`.

In [ ]:
capture_full_output(
    profile_results.print_summary,
    title="Poisson profiling summary",
    include=[r"^setup:", r"^model\.integrate$", r"^prop: ", r"^solve:", r"^update_feec_variables$"],
)


In [ ]:
from scope_profiler import plot_durations, plot_gantt

include_regions = [
    r"^setup:",
    r"^model\.integrate$",
    r"^prop: ",
    r"^solve:",
    r"^kernel: ",
    r"^update_feec_variables$",
]

In [ ]:

duration_figure, _ = plot_durations(
    profile_results,
    include=include_regions,
    sort_by="total",
    top_n=12,
    return_fig=True,
)



In [ ]:
gantt_figure, _ = plot_gantt(
    profile_results,
    include=include_regions,
    return_fig=True,
)

The duration chart ranks regions by aggregate time. The Gantt chart shows when each region executes over wall-clock time. The flame graph reconstructs nested calls, so it is the most useful view for reading parent/child relationships between `setup`, `model.integrate`, propagators, solvers and kernels.

## Cleanup

Remove the temporary directory when you no longer need the profiling files.

In [ ]:
# shutil.rmtree(workdir)